# 7. Protótipo do detector de fraudes

**Projeto:** Agente Identificador de Fraudes Bancárias  
**Disciplina:** Inteligência Artificial – 7º semestre – Ciência da Computação  
**Instituição:** Universidade Presbiteriana Mackenzie – Faculdade de Computação e Informática  
**Professor:** Prof. Dr. Ivan Carlos Alcântara de Oliveira

| Integrante | RA | E-mail |
|---|---|---|
| Cheuk Ki Yu | 10419664 | cheuk.yu@mackenzista.com.br |
| Gabriela Nellessen de Sousa | 10441930 | gabrielanellessen@gmail.com |
| Milton Almeida Leoncio | 10416764 | milton.almeida@live.com |

**Síntese:** Usa o modelo final para avaliar as transações de um dia do período de teste, gerar a fila de alertas e explicar cada alerta. O painel interativo está em prototipo/app.py (streamlit run prototipo/app.py).

**Histórico de alterações**

| Data | Autor | Descrição |
|---|---|---|
| 2026-11-20 | Milton Almeida Leoncio | Versão final do projeto (N2) |

In [1]:
# ========================================================================================
# Projeto: Agente Identificador de Fraudes Bancárias
# Disciplina: Inteligência Artificial - 7º semestre - Ciência da Computação
# Universidade Presbiteriana Mackenzie - Faculdade de Computação e Informática (FCI)
# Professor: Prof. Dr. Ivan Carlos Alcântara de Oliveira
# Integrantes:
#   Cheuk Ki Yu                  - RA 10419664 - cheuk.yu@mackenzista.com.br
#   Gabriela Nellessen de Sousa  - RA 10441930 - gabrielanellessen@gmail.com
#   Milton Almeida Leoncio       - RA 10416764 - milton.almeida@live.com
# Arquivo: notebooks/07_prototipo.ipynb
# Síntese: Usa o modelo final para avaliar as transações de um dia do período de teste, gerar a fila de alertas e explicar cada alerta. O painel interativo está em prototipo/app.py (streamlit run prototipo/app.py).
# Histórico de alterações:
#   2026-11-20 - Milton Almeida Leoncio - Versão final do projeto (N2)
# ========================================================================================

import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
os.environ.setdefault('FRAUDES_ATRIBUTOS', str(RAIZ / 'atributos'))  # saída de preparar_bases.py
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
FIG = RAIZ / 'resultados'

In [2]:
sys.path.insert(0, str(RAIZ / 'prototipo'))
from detector import DetectorFraude, periodo_de_teste
det = DetectorFraude('B')
det

DetectorFraude(Base B - Sparkov, modelo=LightGBM, limiar=0.340, 14 atributos)

In [3]:
fluxo = periodo_de_teste('B')
dia = pd.Timestamp('2020-10-15').date()  # dia do teste com fraudes
lote = fluxo[fluxo['ts'].dt.date == dia]
import time
t0 = time.perf_counter(); r = det.avaliar(lote); ms = 1000 * (time.perf_counter() - t0) / len(lote)
print(f'{dia}: {len(lote)} transações, {int(r.alerta.sum())} alertas, {ms:.3f} ms por transação (com explicações)')
fila = lote.assign(risco=r['escore'], motivos=r['motivos'])[r['alerta']].sort_values('risco', ascending=False)
fila[['ts', 'valor', 'risco', 'is_fraud', 'motivos']]

2020-10-15: 312 transações, 16 alertas, 0.052 ms por transação (com explicações)


,ts,valor,risco,is_fraud,motivos
5810,2020-10-15 00:36:48,290.000,0.999,1,"valor = US$ 290,00 (+7,90) | soma dos valores ..."
5829,2020-10-15 02:24:35,284.760,0.999,1,"valor = US$ 284,76 (+7,93) | soma dos valores ..."
6096,2020-10-15 23:20:27,830.700,0.999,1,"valor = US$ 830,70 (+8,30) | soma dos valores ..."
5803,2020-10-15 00:06:07,17.450,0.998,1,"soma dos valores em 24h = US$ 3.681,70 (+5,57)..."
6047,2020-10-15 20:27:53,863.500,0.998,1,"valor = US$ 863,50 (+8,95) | soma dos valores ..."
5816,2020-10-15 00:51:58,19.040,0.998,1,"soma dos valores em 24h = US$ 3.685,50 (+5,62)..."
5819,2020-10-15 01:24:59,18.460,0.997,1,"soma dos valores em 24h = US$ 3.704,54 (+5,72)..."
5820,2020-10-15 01:37:04,7.230,0.996,1,"soma dos valores em 24h = US$ 4.243,55 (+4,85)..."
5813,2020-10-15 00:42:27,850.160,0.996,1,"valor = US$ 850,16 (+8,21) | soma dos valores ..."
5806,2020-10-15 00:20:49,870.550,0.994,1,"valor = US$ 870,55 (+8,68) | soma dos valores ..."


Cada alerta traz os três atributos que mais aumentaram o risco, em linguagem simples. A decisão de bloquear ou liberar a transação é do analista.

In [4]:
det_a = DetectorFraude('A')
fluxo_a = periodo_de_teste('A')
lote_a = fluxo_a[fluxo_a['ts'].dt.date == fluxo_a['ts'].dt.date.min()]
r_a = det_a.avaliar(lote_a)
print(det_a)
lote_a.assign(risco=r_a['escore'], motivos=r_a['motivos'])[r_a['alerta']].sort_values('risco', ascending=False)[['ts', 'valor', 'risco', 'is_fraud', 'motivos']].head(10)

DetectorFraude(Base A - IBM, modelo=LightGBM, limiar=0.427, 29 atributos)


,ts,valor,risco,is_fraud,motivos
2154,2019-01-01 11:47:00,15.110,0.998,1,"compra no exterior = sim (+8,97) | 1º uso do M..."
2447,2019-01-01 12:32:00,107.000,0.986,1,"compra no exterior = sim (+8,64) | estado ≠ do..."
2703,2019-01-01 13:11:00,58.580,0.980,1,"compra no exterior = sim (+9,81) | estado ≠ do..."
2505,2019-01-01 12:40:00,11.800,0.908,1,"compra no exterior = sim (+10,25) | estado ≠ d..."
1909,2019-01-01 11:10:00,78.000,0.890,1,"compra no exterior = sim (+8,59) | estado ≠ do..."
4014,2019-01-01 17:52:00,29.610,0.886,1,"compra no exterior = sim (+9,13) | 1º uso do M..."
2784,2019-01-01 13:28:00,26.770,0.874,1,"compra no exterior = sim (+8,73) | estado ≠ do..."
2175,2019-01-01 11:50:00,79.350,0.841,1,"compra no exterior = sim (+8,77) | estado ≠ do..."
2340,2019-01-01 12:15:00,1.190,0.800,1,"compra no exterior = sim (+8,51) | estado ≠ do..."
3001,2019-01-01 14:13:00,1.220,0.590,1,"compra no exterior = sim (+8,48) | estado ≠ do..."
